In [10]:
import json
import os
import cv2
from tqdm import tqdm
import shutil
import glob

In [7]:

images_dir = "/data_all/share/models/HuaWei/QC2025发送版/CPRI电源检测"
json_path = "/data_all/share/models/HuaWei/QC2025发送版/CPRI电源检测"
save_dir = "data/cpri/raw_images"
os.makedirs(save_dir,exist_ok = True)

**保存图像到当前项目**

In [ ]:
for root,dirs,files in os.walk(images_dir):
    for file in files:
        if file.endswith("json") == False:
            continue
        # 仅提取有json的
        json_path = os.path.join(root,file)
        with open (json_path,"r",encoding = 'utf-8') as f:
            data = json.load(f)
        image_path = os.path.join(root,data['imagePath'])
        dst_path = os.path.join(save_dir,data['imagePath'])
        shutil.copy(image_path,dst_path)

**构造目标检测类别映射**

In [14]:
classes_map = {}
classes = []
json_folder = "/data_all/share/models/HuaWei/QC2025发送版/CPRI电源检测/*.json"
print(len(glob.glob(json_folder)))
for json_path in glob.glob(json_folder):
    with open (json_path,"r",encoding = 'utf-8') as f:
        data = json.load(f)
    shapes = data['shapes']
    for shape in shapes:
        if shape['label'] not in classes:
            classes.append(shape['label'])

200


In [17]:
print(classes)
for i in range(len(classes)):
    classes_map[classes[i]] = i
print(classes_map)

['cpri', 'power', 'cap_not_screw', 'cap_screw']
{'cpri': 0, 'power': 1, 'cap_not_screw': 2, 'cap_screw': 3}


**保存label为yolo格式**

In [28]:
save_label_dir = "data/cpri/raw_labels"
os.makedirs(save_label_dir,exist_ok = True)

In [36]:
json_folder = "/data_all/share/models/HuaWei/QC2025发送版/CPRI电源检测/*.json"
print(len(glob.glob(json_folder)))
for json_path in tqdm(glob.glob(json_folder)):
    with open (json_path,"r",encoding = 'utf-8') as f:
        data = json.load(f)
    
    image_path = data['imagePath']
    img = cv2.imread(os.path.join(images_dir,image_path))
    H,W = img.shape[:2]
    
    # 写入文件
    img_name_no_suffix = image_path.split('.')[0]
    label_name = img_name_no_suffix + ".txt"
    save_file = os.path.join(save_label_dir,label_name)
    with open (save_file,"w",encoding = 'utf-8') as f:
        shapes = data['shapes']
        for shape in shapes:
            label = classes_map[shape['label']]

            # 提取
            points = shape['points']
            xmin,ymin = int(points[0][0]),int(points[0][1])
            xmax,ymax = int(points[1][0]),int(points[1][1])

            # 转换为 YOLO 格式
            x_center = (xmin + xmax) / 2 / W
            y_center = (ymin + ymax) / 2 / H
            width = (xmax - xmin) / W
            height = (ymax - ymin) / H

            f.write(f"{label} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")


200


100%|██████████| 200/200 [00:06<00:00, 30.00it/s]


**测试是否标对**

In [37]:
def get_four_points(yolo_bbox):
    xc,yc,w,h = yolo_bbox
    xmin = int(xc - w / 2)
    ymin = int(yc - h / 2)
    xmax = int(xc + w / 2)
    ymax = int(yc + h / 2)
    return xmin,ymin,xmax,ymax

In [38]:
img_path = "data/cpri/raw_images/M2T1A746N1003980722896617550.jpg"
label_path = "data/cpri/raw_labels/M2T1A746N1003980722896617550.txt"
raw_json_path = "/data_all/share/models/HuaWei/QC2025发送版/CPRI电源检测/M2T1A746N1003980722896617550.json"
img = cv2.imread(img_path)
img2 = img.copy()
H,W = img.shape[:2]
with open (label_path,"r",encoding = 'utf-8') as f:
    for line in f:
        line = line.strip()
        c,xc,yc,w,h = line.split()
        c = int(c)
        xc = int(W * float(xc))
        yc = int(H * float(yc))
        w = int(W * float(w))
        h = int(H * float(h))
        xmin,ymin,xmax,ymax = get_four_points([xc,yc,w,h])
        cv2.rectangle(img,(xmin,ymin),(xmax,ymax),(0,0,255),2)
    cv2.imwrite("demo.png",img)

with open (raw_json_path,"r",encoding = 'utf-8') as f:
    data = json.load(f)
    shapes = data['shapes']
    for shape in shapes:

        # 提取
        points = shape['points']
        xmin,ymin = int(points[0][0]),int(points[0][1])
        xmax,ymax = int(points[1][0]),int(points[1][1])

        cv2.rectangle(img2,(xmin,ymin),(xmax,ymax),(0,0,255),2)
    cv2.imwrite("demo2.png",img2)